### Second Miletone Project
### Develop By - Dinidu Rukshan
### Date - 2026/08/11


#Load Libries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data Set

In [ ]:
df = pd.read_csv('/content/11_nyc_taxi_fare.csv')

In [ ]:
#head
df.head(5)

#Part A — Exploratory Data Analysis Notebook

In [ ]:
#Shape
#Reason - By using the `shape` command, we can get an idea of the total number of rows and columns in the dataset.
df.shape

In [ ]:
#Columns
# Reason - By using 'columns' command , we can see the all columns of data sheet
df.columns

In [ ]:
#isnull
#reason - by using is null we can check all columns total null records
df.isnull().sum()

##### Missing Value Analysis: The dataset contains missing values in nine columns. The highest number of missing values is found in rate_code with 1,942 missing records, followed by congestion_surcharge with 1,334 missing records. Other affected columns include passenger_count, traffic_index, precipitation_in, weather_condition, store_and_fwd_flag, trip_duration, and fare_amount. I will investigate the percentage and nature of these missing values before selecting an appropriate treatment during the data-cleaning stage.

In [ ]:
plt.figure(figsize=(20, 6))

sns.heatmap(df.isnull(),
            cbar=False,
            yticklabels=False)

plt.title("Missing Value Map")
plt.xlabel("Columns")
plt.ylabel("Rows")

plt.show()

In [ ]:
missing_counts = df.isnull().sum()

missing_counts = missing_counts[missing_counts > 0]

plt.figure(figsize=(10, 6))

missing_counts.sort_values(ascending=False).plot(kind="bar")

plt.title("Missing Values by Column")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")

plt.xticks(rotation=45)
plt.show()

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percentage': (df.isnull().sum() / len(df)) * 100
})

missing_summary = missing_summary[
    missing_summary['missing_count'] > 0
].sort_values('missing_count', ascending=False)

missing_summary

##### Missing Values: The dataset contains missing values in nine columns. rate_code has the highest proportion of missing values at 37.93%, followed by congestion_surcharge at 26.05%. These two columns have substantial missing data and therefore require further investigation before selecting an appropriate treatment. The remaining columns have less than 5% missing values, with fare_amount having only 0.20%. I will investigate the pattern and meaning of these missing values before applying imputation or removal during the data-cleaning stage.

In [ ]:
# infor
#reason - By using 'infor' we can see columns data type
df.info()

##### The dataset contains 5,120 records and 35 columns. There are 19 float64 columns, 1 int64 column, and 15 object columns. Most columns are fully populated, but missing values are present in trip_duration, passenger_count, rate_code, store_and_fwd_flag, weather_condition, precipitation_in, traffic_index, congestion_surcharge, and fare_amount. The rate_code column has the largest number of missing records, with only 3,178 out of 5,120 records containing values. I will investigate these missing values and review the data types of the object columns before modelling.

In [ ]:
df.describe()

### Summary of Data Insights from `df.describe()`

From the `df.describe()` outputs, we can gather several key insights about your dataset:

#### For Numerical Columns (`df.describe()`):

*   **Count:** Shows the number of non-null entries for each column, indicating potential missing values. For example, `passenger_count`, `precipitation_in`, `traffic_index`, `congestion_surcharge`, and `fare_amount` have missing values.
*   **Mean, Std, Min, Max, Quartiles (25%, 50%, 75%):** These statistics help understand the distribution and spread of the data.
    *   **`driver_experience_years`**: Ranges from 0.1 to 36.6 years, with an average of about 6.79 years.
    *   **Geographical Coordinates (`pickup_latitude`, `pickup_longitude`, `dropoff_latitude`, `dropoff_longitude`):** The min/max values for `pickup_latitude` (-74.16) and `dropoff_longitude` (0.00) seem unusual and might indicate data entry errors or incorrect coordinates.
    *   **`trip_distance`**: Ranges from 0 to 53.83 miles, with an average of around 11.39 miles. The minimum of 0 might indicate short or cancelled trips.
    *   **`passenger_count`**: Ranges from 0 to 9. The average is about 1.75 passengers, and the minimum of 0 might also indicate issues or empty trips.
    *   **`temperature_f`**: Shows a minimum value of -999, which is clearly an outlier or an indicator for missing data, suggesting further data cleaning is needed for this column.
    *   **`fare_amount`**: Has a minimum value of -85.09, which is highly unusual for a fare and indicates data quality issues that need to be addressed.
    *   **`tolls_amount`, `extra_surcharge`, `mta_tax`, `improvement_surcharge`, `congestion_surcharge`**: These seem to be mostly standard charges or small amounts, with `tolls_amount` having a wider range.



### Structural Inspection: Duplicate Check

In [ ]:
df.duplicated().any()

### np.True_ meaning of Duplicate rows is the data sheet , Now we can see the duplcate row count

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

######Duplicate Record Analysis: The dataset contains 120 duplicate records. I will investigate these duplicate rows to determine whether they are genuine repeated transactions or data-entry duplicates before removing them.

# Target Analysis

##### Target Varible is -  fare_amount
The target is examined for distribution shape, skewness, and outliers.
A transformation decision is then made based on quantitative evidence.

In [ ]:
df.head(5)

In [ ]:
y = df['fare_amount']

neg  = df[y < 0]
zero = df[y == 0]

print(f"Negative fares : {len(neg)}")
print(f"Zero fares     : {len(zero)}")
print(f"Missing        : {y.isna().sum()}\n")

if len(neg):
    print("--- Negative fare records ---")
    print(neg[['fare_amount', 'trip_distance', 'tolls_amount',
               'tip_amount', 'passenger_count']])

In [ ]:
from scipy import stats


fig, ax = plt.subplots(2, 2, figsize=(14, 9))

sns.histplot(y.dropna(), bins=80, kde=True, ax=ax[0,0])
ax[0,0].axvline(y.mean(),   color='red',    ls='--', label=f'Mean {y.mean():.1f}')
ax[0,0].axvline(y.median(), color='green',  ls='--', label=f'Median {y.median():.1f}')
ax[0,0].set_title('Distribution of fare_amount')
ax[0,0].legend()

sns.boxplot(x=y, ax=ax[0,1])
ax[0,1].set_title('Boxplot — outlier exposure')

stats.probplot(y.dropna(), dist='norm', plot=ax[1,0])
ax[1,0].set_title('Q-Q Plot — raw target')

sns.histplot(y[y.between(0, 150)], bins=60, kde=True, ax=ax[1,1])
ax[1,1].set_title('Zoomed: 0-150 (main body)')

plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

# Ensure y is defined as the target variable 'fare_amount'
y = df['fare_amount'].dropna()

# Filter for positive fares for log, sqrt, and box-cox transformations
y_positive = y[y > 0]

# Define candidate transformations for comparison
candidates = {
    'raw': y,
    'log': np.log(y_positive),
    'sqrt': np.sqrt(y_positive),
    'boxcox': pd.Series(stats.boxcox(y_positive)[0], index=y_positive.index, name='boxcox')
}

fig, ax = plt.subplots(2, 4, figsize=(18, 8))

for i, (name, s) in enumerate(candidates.items()):
    sns.histplot(s, bins=60, kde=True, ax=ax[0, i])
    ax[0, i].set_title(f'{name}\nskew = {s.skew():.3f}')
    stats.probplot(s, dist='norm', plot=ax[1, i])
    ax[1, i].set_title(f'Q-Q: {name}')

plt.tight_layout()
plt.show()

In [ ]:
df['fare_amount'].skew()

#### Conclusion on Target Variable Transformation:

The analysis of the `fare_amount` distribution and its various transformations (logarithmic, square root, and Box-Cox) reveals that:

*   The **raw `fare_amount`** is highly right-skewed (skewness = 7.941), with a large number of lower fares and a few extremely high fares, and significant deviation from normality as shown by the Q-Q plot.
*   Both **logarithmic** (skewness = 0.084) and **square root** (skewness = 1.991) transformations improve the normality and reduce skewness, making the distributions more symmetric.
*   The **Box-Cox transformation** is the most effective, resulting in a distribution with a skewness of -0.004, which is very close to zero. Its Q-Q plot aligns most closely with the normal line.

Therefore, to meet the assumptions of many statistical and machine learning models that prefer normally distributed target variables, the **Box-Cox transformation will be applied to the `fare_amount`** in the data preparation phase. This will help in creating a more robust and accurate predictive model.

In [ ]:
# Calculate IQR and bounds for outlier detection
q1, q3 = y.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Identify outliers based on IQR
out_iqr = y[(y < lo) | (y > hi)]

# Calculate Z-scores and identify outliers
z = np.abs(stats.zscore(y.dropna()))

print(f"IQR bounds     : {lo:.2f}  to  {hi:.2f}")
print(f"IQR outliers   : {len(out_iqr)}  ({len(out_iqr) / len(y) * 100:.2f}%)")
print(f"Z-score > 3    : {(z > 3).sum()}  ({(z > 3).sum() / len(y.dropna()) * 100:.2f}%)\n")

print("Upper percentiles of fare_amount:")
print(y.quantile([0.90, 0.95, 0.99, 0.995, 0.999, 1.0]))

#### Conclusion on Outliers:

The outlier analysis using both the Interquartile Range (IQR) method and Z-score method confirms the presence of a significant number of outliers in the `fare_amount` target variable.

*   **IQR Method**: Identifies `168` outliers, representing `3.37%` of the dataset. The bounds are from `-33.31` to `121.53`. Notably, this method captures both the negative fare amounts (as values below the lower bound) and extremely high fare amounts.

*   **Z-score Method**: Identifies `171` outliers, representing `3.43%` of the dataset, when using a threshold of `|Z| > 3`. This method is effective for normally distributed data, but given the skewed nature of the raw `fare_amount`, its interpretation should be cautious.

*   **Upper Percentiles**: The quantiles show a steep increase at the higher end, with the 99th percentile being `176.54`, and the maximum value reaching `986.60`. This further highlights the presence of extreme values.

**Action Plan**: The presence of these outliers, especially the negative `fare_amount` values and extremely high positive values, indicates data quality issues that need to be addressed. In the data cleaning phase, I will develop a strategy to handle these outliers, which may involve capping, transformation, or removal, depending on their impact on the model and the context of the data.

In [ ]:
num_cols = ['trip_distance', 'trip_duration', 'passenger_count',
            'tolls_amount', 'traffic_index', 'driver_experience_years',
            'temperature_f', 'precipitation_in']

cat_cols = ['pickup_borough', 'dropoff_borough', 'payment_type',
            'rate_code', 'vehicle_type', 'weather_condition', 'is_holiday']

### Univariate Analysis of Key Predictors

> **Note on scope:** `trip_duration` is stored as free text (for example `"1h 20m"`) rather
> than as a number, so it cannot be plotted or correlated in its raw state. The conversion
> below is applied here only to make the exploratory plots possible; it is itself one of the
> data quality defects catalogued in this notebook and is implemented properly in the Part B
> cleaning pipeline.

In [ ]:
def convert_duration_to_minutes(duration_str):
    if pd.isna(duration_str):
        return np.nan
    duration_str = str(duration_str).strip()
    total_minutes = 0
    if 'h' in duration_str:
        parts = duration_str.split('h')
        hours = int(parts[0].strip())
        total_minutes += hours * 60
        if 'm' in parts[1]:
            minutes = int(parts[1].replace('m', '').strip())
            total_minutes += minutes
    elif 'm' in duration_str:
        total_minutes = int(duration_str.replace('m', '').strip())
    else:
        try:
            # Handle cases that might already be pure numbers as strings
            return float(duration_str)
        except ValueError:
            return np.nan # Or raise an error for unhandled formats
    return float(total_minutes)

df['trip_duration'] = df['trip_duration'].apply(convert_duration_to_minutes)

fig, ax = plt.subplots(len(num_cols), 2, figsize=(12, 4*len(num_cols)))

for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), bins=50, kde=True, ax=ax[i, 0])
    ax[i, 0].set_title(f'{col} — distribution (skew = {df[col].skew():.2f})')

    sns.boxplot(x=df[col], ax=ax[i, 1])
    ax[i, 1].set_title(f'{col} — outliers')

plt.tight_layout()
plt.show()

In [ ]:
# ---- Data quality defect: sentinel value hiding inside temperature_f ----
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

sns.histplot(df['temperature_f'].dropna(), bins=60, ax=ax[0], color='indianred')
ax[0].set_title(f"temperature_f AS STORED  (skew = {df['temperature_f'].skew():.2f})")
ax[0].set_xlabel('temperature_f')

sns.boxplot(x=df['temperature_f'], ax=ax[1], color='indianred')
ax[1].set_title('Boxplot — note the isolated cluster at -999')

valid = df.loc[df['temperature_f'] != -999, 'temperature_f']
sns.histplot(valid.dropna(), bins=60, ax=ax[2], color='seagreen')
ax[2].set_title(f'temperature_f EXCLUDING -999  (skew = {valid.skew():.2f})')
ax[2].set_xlabel('temperature_f')

plt.tight_layout()
plt.show()

n_sentinel = (df['temperature_f'] == -999).sum()
print(f"Records with temperature_f == -999 : {n_sentinel} "
      f"({n_sentinel / len(df) * 100:.2f}%)")
print(f"Valid temperature range            : "
      f"{valid.min():.1f} F to {valid.max():.1f} F")
print(f"Skewness as stored                 : {df['temperature_f'].skew():.3f}")
print(f"Skewness once -999 is excluded     : {valid.skew():.3f}")

print("\nImpossible values found in other predictors:")
print(f"  trip_distance   == 0 : {(df['trip_distance'] == 0).sum()} records")
print(f"  passenger_count == 0 : {(df['passenger_count'] == 0).sum()} records")

#### Conclusion on Univariate Distributions and the Sentinel-Value Defect:

The univariate plots expose a clear data quality defect in `temperature_f`. The histogram and
boxplot above show an isolated cluster of observations at exactly **-999**, separated by a wide
empty gap from the genuine readings, which fall between 1.0 F and 105.0 F. A temperature of
-999 F is physically impossible, so this is not an outlier in any statistical sense: it is a
**sentinel value** written by the source system to mark a missing weather reading. Because it is
stored as a valid number rather than as `NaN`, `df.isnull().sum()` reports zero missing values
for this column and the defect is invisible to the missing-value analysis performed earlier.

The distortion is severe. With the sentinel values included the column has a skewness of
**-10.789**; once the 37 affected records are excluded the skewness falls to **-0.125**, which is
very close to symmetric. The mean is dragged downward by roughly seven degrees. Any model
trained on this column as stored would be learning from a fabricated value.

I will therefore replace all `-999` entries in `temperature_f` with `NaN` in Part B and then
impute them using the median, so that they are treated as genuinely missing rather than as real
observations.

Two further impossibilities appear in the same set of plots. **88 records report a
`trip_distance` of exactly 0** and **64 records report a `passenger_count` of 0**, yet all of
these rows carry a positive fare. A journey of zero miles carrying zero passengers cannot
generate a charge, so these represent cancelled trips or meter errors. I will investigate and
remove or correct them during the Part B cleaning stage.

Regarding distribution shape, the remaining predictors are moderately right-skewed:
`tolls_amount` (skew = 2.36), `passenger_count` (1.99), `driver_experience_years` (1.36),
`trip_distance` (1.19) and `trip_duration` (1.11). Only `traffic_index` (0.23) is close to
symmetric. The skew in `tolls_amount` and `precipitation_in` (2.93) is structural rather than
problematic, since most trips pay no toll and most days record no rainfall, producing a large
spike at zero. I will retain these as-is but will consider binary indicator features
(`has_toll`, `is_wet`) in Part B to capture that zero-inflation more directly.

### Data Quality Defect: Geographic Coordinates

The sentinel-value analysis above covers `temperature_f`. The same class of defect is
present in the four coordinate columns, in two different forms, and neither is visible
to `df.isnull().sum()`.

In [ ]:
# ---- Data quality defect: coordinates ----
coord_cols = ['pickup_latitude', 'pickup_longitude',
              'dropoff_latitude', 'dropoff_longitude']

# NYC is bounded by roughly lat 40.4-41.0 and lon -74.3 to -73.6
NYC_LAT = (40.4, 41.0)
NYC_LON = (-74.3, -73.6)

print("Defect 1 - missing coordinates stored as 0.0 ('Null Island', 0N 0E):")
for c in coord_cols:
    n = (df[c] == 0).sum()
    print(f"  {c:20s} == 0 : {n:4d} records")

print("\nDefect 2 - latitude and longitude written into the wrong column:")
for p in ['pickup', 'dropoff']:
    lat, lon = f'{p}_latitude', f'{p}_longitude'
    swapped = ((df[lat] < 0) & (df[lon] > 0)).sum()
    print(f"  {p:8s}: {swapped:4d} records have a negative latitude "
          f"and a positive longitude")

print("\nStored ranges (a NYC latitude can never be negative):")
print(df[coord_cols].agg(['min', 'max']).round(4))

In [ ]:
# visual evidence: plotting the raw coordinates shows the defects immediately
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

ax[0].scatter(df['pickup_longitude'], df['pickup_latitude'], s=4, alpha=0.3)
ax[0].set_title('Pickup coordinates AS STORED')
ax[0].set_xlabel('longitude'); ax[0].set_ylabel('latitude')

ok = (df['pickup_latitude'].between(*NYC_LAT) &
      df['pickup_longitude'].between(*NYC_LON))
ax[1].scatter(df.loc[ok, 'pickup_longitude'], df.loc[ok, 'pickup_latitude'],
              s=4, alpha=0.3, color='seagreen')
ax[1].set_title('Pickup coordinates WITHIN NYC BOUNDS')
ax[1].set_xlabel('longitude'); ax[1].set_ylabel('latitude')

plt.tight_layout()
plt.show()

outside = (~ok).sum()
print(f"Records outside plausible NYC bounds: {outside} "
      f"({outside / len(df) * 100:.2f}%)")

#### Conclusion on the Coordinate Defects:

Two distinct defects are present in the coordinate columns, and neither is reported by
`df.isnull().sum()` because both are stored as valid numbers.

The first is the same sentinel pattern already identified in `temperature_f`: missing
coordinates have been written as `0.0`. The point `0N 0E` lies in the Atlantic Ocean off
the coast of Africa and cannot represent a New York taxi trip.

The second is a **column transposition**. A number of records hold a negative value in
the latitude field and a positive value in the longitude field, which is the reverse of
the correct signs for New York. Latitude in this city is always positive and in the range
40.4 to 41.0; longitude is always negative and in the range -74.3 to -73.6. These rows have
had the two fields written the wrong way round at the source.

Both defects are corrected in the Part B cleaning pipeline: zeros are converted to `NaN`,
transposed pairs are swapped back, and any remaining coordinate outside the city bounds
is treated as missing.

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(16, 12))
ax = ax.flatten()

for i, col in enumerate(cat_cols):
    df[col].value_counts(dropna=False).plot(kind='bar', ax=ax[i])
    ax[i].set_title(f'{col} — value counts')
    ax[i].tick_params(axis='x', rotation=45)

for j in range(len(cat_cols), len(ax)):
    ax[j].axis('off')

plt.tight_layout()
plt.show()

#### Conclusion on Categorical Label Inconsistency:

The value-count bar charts above reveal that three categorical columns carry the same real-world
category under multiple different labels, which inflates the apparent number of levels and would
fragment any encoding applied during modelling.

`payment_type` displays **seven** distinct labels where only **four** categories exist:
`CRD` and `Credit Card` are the same payment method, `CSH` and `Cash` are the same method, and
`No Charge` and `No charge` differ only in capitalisation. The split is material rather than
cosmetic: the credit-card category is divided into groups of 732 and 2,544 records, so
one-hot encoding this column as it stands would create two sparse, redundant columns instead of
one informative one.

`weather_condition` displays **six** labels where only **four** conditions exist, with `RAIN`
and `Rain` separated (89 and 1,049 records) and `Clear` and `clear` separated (2,964 and 238
records). This is a pure casing inconsistency.

`rate_code` is the most seriously affected. It mixes **numeric codes with their own text
descriptions in the same column**: the values `1`, `2`, `3`, `4` and `5` appear alongside
`Standard Rate`, `JFK Flat Fare`, `Newark`, `Nassau/Westchester` and `Negotiated`. The group
statistics confirm these are duplicates of one another rather than distinct rate classes, since
code `1` (mean fare 35.05) and `Standard Rate` (mean fare 40.30) describe the same tariff, as do
code `4` (163.73) and `Nassau/Westchester` (167.49). Ten apparent levels therefore collapse to
roughly five or six. This also explains the 37.93% missing rate observed earlier, since the
column is evidently being populated by two different upstream systems.

In Part B I will standardise all three columns by lower-casing and stripping whitespace, then
applying an explicit mapping dictionary to merge the abbreviations with their full labels and to
convert the numeric `rate_code` values to their text equivalents, before any encoding is
performed.

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(16, 12))
ax = ax.flatten()

for i, col in enumerate(num_cols):
    sns.scatterplot(x=df[col], y=df['fare_amount'], alpha=0.3, ax=ax[i])
    r = df[[col, 'fare_amount']].corr().iloc[0, 1]
    ax[i].set_title(f'{col} vs fare_amount  (r = {r:.2f})')

for j in range(len(num_cols), len(ax)):
    ax[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(16, 12))
ax = ax.flatten()

for i, col in enumerate(cat_cols):
    sns.boxplot(x=df[col], y=df['fare_amount'], ax=ax[i])
    ax[i].set_title(f'fare_amount by {col}')
    ax[i].tick_params(axis='x', rotation=45)

for j in range(len(cat_cols), len(ax)):
    ax[j].axis('off')

plt.tight_layout()
plt.show()

# numeric summary to support the plots
for col in cat_cols:
    print(df.groupby(col)['fare_amount'].agg(['count', 'mean', 'median']), '\n')

#### Conclusion on Bivariate Relationships:

The bivariate analysis shows that geography is the strongest categorical driver of fare. Mean
fare by pickup borough ranges from 39.94 in Brooklyn and 41.97 in Manhattan up to 81.92 in New
Jersey and **165.02 in Staten Island** — roughly four times the Manhattan average. The pattern
repeats on the drop-off side, where Staten Island averages 164.46 against 38.34 for Manhattan.
This is consistent with the geography of the city, since Staten Island and New Jersey trips
cross water and cover far longer distances. Both borough columns are clearly informative and I
will retain them as encoded features.

`rate_code` shows the same discriminating power once its label inconsistency is accounted for,
with `Nassau/Westchester` averaging 167.49 and code `4` averaging 163.73 against 35.05 for the
standard rate. This confirms the column is worth repairing rather than discarding, despite its
high missing rate.

By contrast, several columns show almost no relationship with the target. `vehicle_type` varies
only between 45.71 and 49.85 across all four categories, and `is_holiday` varies between 47.43
and 52.97 — differences far smaller than the within-group spread shown by the boxplots. The
numeric weather and traffic predictors behave the same way, with `temperature_f`, `traffic_index`
and `precipitation_in` all producing flat, formless scatter clouds against `fare_amount`.

I will therefore prioritise the distance, duration, borough and rate-code features in Part B, and
will test whether the weather and traffic columns can be dropped entirely without loss of
predictive performance.

In [ ]:
# guard: only strip '$' if the column is not already numeric, so re-running this cell is safe
if not pd.api.types.is_numeric_dtype(df['total_amount']):
    df['total_amount'] = (df['total_amount'].astype(str)
                          .str.replace('$', '', regex=False)
                          .str.replace(',', '', regex=False)
                          .astype(float))

corr_cols = num_cols + ['tip_amount', 'total_amount', 'fare_amount']
corr = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=.5)
plt.title('Correlation Matrix — numeric features vs fare_amount')
plt.show()

In [ ]:
print("Correlation with fare_amount (sorted):")
print(corr['fare_amount'].drop('fare_amount').sort_values(ascending=False))

### Correlation Re-measured on Clean Data

The correlation matrix above is computed on the data **as stored**, which still contains
the `-999` sentinel in `temperature_f`, the negative fare records, and the extreme upper
outliers. A correlation measured through that much corruption cannot be trusted, so the
same statistics are recomputed below on a cleaned copy for comparison. The original `df`
is not modified.

In [ ]:
# cleaned COPY for measurement only - df itself is left untouched
df_clean = df.copy()

df_clean['temperature_f'] = df_clean['temperature_f'].replace(-999, np.nan)
df_clean[coord_cols] = df_clean[coord_cols].replace(0, np.nan)

# remove records that cannot physically exist
df_clean = df_clean[(df_clean['fare_amount'] > 0) &
                    (df_clean['trip_distance'] > 0) &
                    (df_clean['passenger_count'] > 0)]

# trim the extreme upper tail identified in the outlier analysis
hi = df_clean['fare_amount'].quantile(0.995)
df_clean = df_clean[df_clean['fare_amount'] <= hi]

print(f"rows: {len(df)} (as stored)  ->  {len(df_clean)} (cleaned copy)")

corr_clean = df_clean[corr_cols].corr()

comparison = pd.DataFrame({
    'as_stored': corr['fare_amount'].drop('fare_amount'),
    'cleaned':   corr_clean['fare_amount'].drop('fare_amount')
})
comparison['change'] = (comparison['cleaned'] - comparison['as_stored'])
print("\nCorrelation with fare_amount - before and after cleaning:")
print(comparison.sort_values('cleaned', ascending=False).round(3))

#### Conclusion on the Effect of Cleaning on Correlation:

The comparison table shows how much the data quality defects distorted the measured
relationships. Correlations computed on the uncleaned data understate the true strength
of the genuine predictors, because the negative fares, the extreme upper outliers and the
`-999` sentinel all add variance that is unrelated to any real signal.

This matters for feature selection. A predictor dismissed as uninformative on the basis of
a correlation measured through corrupted data may in fact be useful once the corruption is
removed, so **feature selection decisions are deferred until after cleaning** rather than
being made from the figures in the previous section.

In [ ]:
c = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print("Highly correlated pairs (|r| > 0.7):")
print(c[c > 0.7].sort_values(ascending=False))

### Target Leakage Check

Before any feature is carried into modelling, each column is tested against a single
question: *would this value be known at the moment the prediction has to be made?*
A column that is derived from the target, or that is only recorded after the trip has
finished, cannot be used as a predictor however strongly it correlates.

In [ ]:
# ---- Target leakage audit ----
leakage_candidates = {
    'total_amount':          'fare_amount + tip + tolls + surcharges - CONTAINS the target',
    'tip_amount':            'recorded after the trip ends',
    'tolls_amount':          'recorded after the trip ends',
    'extra_surcharge':       'billing component added to the fare',
    'mta_tax':               'billing component added to the fare',
    'improvement_surcharge': 'billing component added to the fare',
    'congestion_surcharge':  'billing component added to the fare',
    'trip_rating':           'submitted after the trip ends',
}

print(f"{'column':<24} {'r with fare':>12}   reason")
print("-" * 78)
for c, reason in leakage_candidates.items():
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
        r = df[[c, 'fare_amount']].corr().iloc[0, 1]
        print(f"{c:<24} {r:>12.3f}   {reason}")
    elif c in df.columns:
        print(f"{c:<24} {'(non-numeric)':>12}   {reason}")

print("\nAll columns listed above are EXCLUDED from the feature set in Part B.")

#### Conclusion on Target Leakage:

`total_amount` is the clearest case. It is defined as
`fare_amount + tip_amount + tolls_amount + surcharges`, so it contains the target inside
it by construction. Its appearance in the highly-correlated pairs table above is therefore
not evidence of a useful predictor; it is evidence of leakage. A model given this column
would report near-perfect accuracy while having learned nothing about what determines a
fare, and would fail immediately on live data where the total is not yet known.

The remaining billing components and `trip_rating` are excluded on timing grounds rather
than by construction: none of them exists at the moment a fare estimate is required.

`trip_duration` occupies a middle position. It correlates strongly with the target
(r = 0.634) and is legitimate for a model that predicts the fare **after** the trip, but
it is unavailable to a model that predicts the fare **at pickup**. The prediction point is
therefore stated explicitly in Part B, and the feature set is chosen to match it.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# use the cleaned copy so the -999 sentinel does not distort the variance
X = df_clean[num_cols].dropna()
X = sm.add_constant(X)

vif = pd.DataFrame({
    'feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif[vif.feature != 'const'].sort_values('VIF', ascending=False))

##### Conclusion on Correlation and Multicollinearity:

trip_duration (r = 0.634) and trip_distance (r = 0.588) show the strongest genuine relationships with fare_amount, followed by tolls_amount (r = 0.425). In contrast, traffic_index (r = -0.023), passenger_count (r = -0.026), temperature_f (r = 0.002) and precipitation_in (r = -0.010) show almost no linear relationship with the target, so I do not expect them to contribute much predictive value.

total_amount (r = 0.752) and tip_amount (r = 0.447) are components of the final charge and are therefore derived from the target itself. Including them would constitute data leakage, so I will remove both before modelling in Part B.

The VIF analysis (computed with an intercept term) returns a maximum of 4.01 for trip_duration and 3.70 for trip_distance, both below the conventional threshold of 5. Multicollinearity is therefore present but not severe. However, the pairwise correlation between these two variables is r = 0.811, so rather than dropping either, I will engineer an average_speed feature in Part B to capture the relationship more efficiently.